## 用于模拟用户端的测试代码
### 初始化

In [1]:
import requests
url = "http://localhost:8000"


### 健康检查

In [2]:
try:
    response = requests.get(f"{url}/healthz")
    if response.status_code == 200:
        print("Health check passed!")
    else:
        raise Exception(f"Health check failed with status code: {response.status_code}")
except Exception as e:
    print(f"An error occurred: {e}")


Health check passed!


In [ ]:
seed = 42
try:
    response = requests.post(f"{url}/seed", json={"seed": seed})
    if response.status_code == 200:
        print(f"Seed set successfully: {response.json()}")
    else:
        raise Exception(f"Failed to set seed with status code: {response.status_code}")
except Exception as e:
    print(f"An error occurred: {e}")

### 参数解析
尽管参数提交给后端时并不区分decode_args和messages，但为了更清晰地表达参数的含义，我们在这里将它们分开说明：
decode_args:
    "task_type": "speech"||"text" // 任务类型，text调用MingAudio.generation，speech调用MingAudio.speech_generation，大多数时候只需要设置为speech
    "response_mode": "path"||"stream" // 响应模式，path表示返回文件路径，stream表示返回音频流
    "output_wav_path": url路径 // 当response_mode为path时，指定输出文件的路径,若为空则自动生成不重复的文件名
    "max_decode_steps": int // 生成的最大步数，超过这个步数后模型将停止生成，默认为200

messages:
    "prompt": "文本提示" // 生成任务的文本提示，一般时用来指导模型生成不同类型的内容
    "text": "生成文本" // 生成任务的文本内容，必填项
    "prompt_wav_path": url路径||list // 参考音频的路径，若不提供则默认为空
    "prompt_text": str||list // 参考音频的文本内容，若不提供则默认为空
    "use_spk_emb": bool // 是否使用说话人嵌入，默认为False，若为True则会使用prompt_wav_path中的音频提取说话人特征来指导生成
    "use_zero_spk_emb": bool // 是否使用零说话人嵌入，默认为False，若为True则会使用全零的说话人特征来指导生成，这在某些情况下可以提高生成的多样性
    "cfg": float
    "temperature": float
    "sigma": float
     // 上面三个参数都是用来控制生成的多样性和质量的，
    "instruction": "附加说明" // 生成任务的附加说明，默认为空,也可以用来传递感情,如下面EMOTION演示中使用了"情感":"高兴"

attr: //BGM生成的属性参数，主要用来指导模型生成符合特定属性的音乐，建议采用演示的写法
    "Genre": str // 音乐的风格
    "Mood": str // 音乐的情绪
    "Instrument": str // 音乐的主要乐器
    "Theme": str // 音乐的主题
    "Duration": str // 音乐的时长
    "SNR": str // 音乐的信噪比
    "ENV": str // 音乐的环境
instuction: 目前已知可以使用的键值对,具体见cookbooks/instructions.md
    "情感": str // 生成语音的情感，如高兴、悲伤、愤怒等
    "语速": str // 生成语音的语速，如快速、中速、慢速等
    "基频": str // 生成语音的基频，如低、中、高等
    "音量": str // 生成语音的音量，如小、中、大等
    "方言": str // 生成语音的方言，如粤语等
    "风格": str // 自然语言描述的声音风格
    "IP": str // 模型内置的角色IP形象
    "BGM": dict // 生成语音的背景音乐属性，包含Genre、Mood、Instrument、Theme、Duration、SNR、ENV等键值对

### 演示

In [ ]:
# TTA
decode_args = {
        "max_decode_steps": 200,
        "cfg": 4.5,
        "sigma": 0.3,
        "temperature": 2.5
}
messages = {
        "prompt": "Please generate audio events based on given text.\n",
        "text": "Thunder and a gentle rain",
}
response = requests.post(url+"/v1/generate", json={**messages, **decode_args})
print("Status Code:", response.status_code)

In [4]:
# Zero-shot TTS
decode_args = {
    "task_type": "speech",
    "response_mode": "path",
    "output_wav_path": "output/service/client_test.wav",
    "max_decode_steps": 200,
}
messages = {
    "prompt": "Please generate speech based on the following description.\n",
    "text": "我们的愿景是构建未来服务业的数字化基础设施，为世界带来更多微小而美好的改变。",
    "use_spk_emb": True,
    "prompt_wav_path": "data/wavs/10002287-00000094.wav",
    "prompt_text": "在此奉劝大家别乱打美白针。"
}
response = requests.post(url+"/v1/generate", json={**messages, **decode_args})
print("Status Code:", response.status_code)

Status Code: 200


In [ ]:
# BGM
decode_args = {
    "max_decode_steps": 400,
}
attr = {
    "Genre": "电子舞曲.",
    "Mood": "自信 / 坚定.",
    "Instrument": "架子鼓.",
    "Theme": "节日.",
    "Duration": "30s."
}
text = " " + " ".join([f"{key}: {value}" for key, value in attr.items()])
messages = {
    "prompt": "Please generate music based on the following description.\n",
    "text": text,
}
response = requests.post(url+"/v1/generate", json={**messages, **decode_args})
print("Status Code:", response.status_code)

In [ ]:
# Emotion
decode_args = {
    "max_decode_steps": 200,
}
instruction = {
    "情感": "高兴"
}
messages = {
    "prompt": "Please generate speech based on the following description.\n",
    "text": "我竟然抢到了陈奕迅的演唱会门票！太棒了！终于可以现场听一听他的歌声了！",
    "use_spk_emb": True,
    "instruction": instruction,
    "prompt_wav_path": "data/wavs/emotion_prompt.wav",
}
response = requests.post(url+"/v1/generate", json={**messages, **decode_args})
print("Status Code:", response.status_code)

In [3]:
# Podcast
decode_args = {
    "max_decode_steps": 200,
}
dialog = [
    {"speaker_1": "你可以说一下，就大概说一下，可能虽然我也不知道，我看过那部电影没有。"},
    {"speaker_2": "就是那个叫什么，变相一节课的嘛。"},
    {"speaker_1": "嗯。"},
    {"speaker_2": "一部搞笑的电影。"},
    {"speaker_1": "一部搞笑的。"}
]
text = " " + "\n ".join([f"{k}:{v}" for item in dialog for k, v in item.items()]) + "\n"
prompt_diag = [
    {"speaker_1": "并且我们还要进行每个月还要考核 笔试的话还要进行笔试，做个，当服务员还要去笔试了"},
    {"speaker_2": "对啊，这真的很奇怪，就是 单纯的因，单纯自己工资不高，只是因为可能人家那个店比较出名一点，就对你苛刻要求"},
]
prompt_text = " " + "\n ".join([f"{k}:{v}" for item in prompt_diag for k, v in item.items()]) + "\n"

messages = {
    "prompt": "Please generate speech based on the following description.\n",
    "text": text,
    "use_spk_emb": True,
    "prompt_wav_path": [
        "data/wavs/CTS-CN-F2F-2019-11-11-423-012-A.wav",
        "data/wavs/CTS-CN-F2F-2019-11-11-423-012-B.wav"
    ],
    "prompt_text": prompt_text
}
response = requests.post(url+"/v1/generate", json={**messages, **decode_args})
print("Status Code:", response.status_code)

Status Code: 200


In [ ]:
# Basic
decode_args = {
    "max_decode_steps": 200,
}
instruction = {
    "语速": "快速",
    "基频": "中",
    "音量": "中",
}
messages = {
    "prompt": "Please generate speech based on the following description.\n",
    "text": "简单地说，这相当于惠普把消费领域市场拱手相让了。",
    "use_spk_emb": True,
    "instruction": instruction,
    "prompt_wav_path": "data/wavs/10002287-00000095.wav",
}
response = requests.post(url+"/v1/generate", json={**messages, **decode_args})
print("Status Code:", response.status_code)

In [ ]:
# Dialect
decode_args = {
    "max_decode_steps": 200
}
instruction = {
    "方言": "广粤话"
}
messages = {
    "prompt": "Please generate speech based on the following description.\n",
    "text": "我觉得社会企业同个人都有责任",
    "use_spk_emb": True,
    "instruction": instruction,
    "prompt_wav_path": "data/wavs/yue_prompt.wav",
}
response = requests.post(url+"/v1/generate", json={**messages, **decode_args})
print("Status Code:", response.status_code)

In [ ]:
# Timbre Definition
decode_args = {
    "task_type": "speech",
    "max_decode_steps": 200,
    "response_mode": "path",
    "output_wav_path": "output/service/client_test_timbre.wav",
}
instruction = {
    "风格": "这是一种ASMR耳语，属于一种旨在引发特殊感官体验的创意风格。这个女性使用轻柔的普通话进行耳语(在一侧耳边)，声音气音成分重。音量极低，紧贴麦克风，语速极慢，旨在制造触发听者颅内快感的声学刺激。"
}
messages = {
    "prompt": "Please generate speech based on the following description.\n",
    "text": "我会一直在这里陪着你，直到你慢慢、慢慢地沉入那个最温柔的梦里……好吗？",
    "instruction": instruction,
    "use_zero_spk_emb": True
}
response = requests.post(url+"/v1/generate", json={**messages, **decode_args})
print("Status Code:", response.status_code)

In [ ]:
# IP
decode_args = {
    "max_decode_steps": 200,
}
instruction = {
    "IP": "灵小甄"
}
messages = {
    "prompt": "Please generate speech based on the following description.\n",
    "text": "这款产品的名字，叫变态坑爹牛肉丸。",
    "instruction": instruction,
    "use_zero_spk_emb": True
}
response = requests.post(url+"/v1/generate", json={**messages, **decode_args})
print("Status Code:", response.status_code)

In [ ]:
# Speech + bgm
decode_args = {
    "max_decode_steps": 200,
}
instruction = {
    "BGM": {"Genre": "当代古典音乐.", "Mood": "温暖 / 友善.", "Instrument": "电吉他", "Theme": "节日.", "SNR": 10.0, "ENV": None},
}
messages = {
    "prompt": "Please generate speech based on the following description.\n",
    "text": "此次业绩下滑原因，可归结为企业停止服务某些品牌，而带来的负面影响。",
    "use_spk_emb": True,
    "instruction": instruction,
    "prompt_wav_path": "data/wavs/00000309-00000300.wav",
}
response = requests.post(url+"/v1/generate", json={**messages, **decode_args})
print("Status Code:", response.status_code)

In [ ]:
# Speech+sound
decode_args = {
    "max_decode_steps": 200,
}
instruction = {
    "BGM": {"ENV": "Birds chirping", "SNR": 10.0, "Genre": None, "Mood": None, "Instrument": None, "Theme": None}
}
messages = {
    "prompt": "Please generate speech based on the following description.\n",
    "text": "此次业绩下滑原因，可归结为企业停止服务某些品牌，而带来的负面影响。",
    "use_spk_emb": True,
    "instruction": instruction,
    "prompt_wav_path": "data/wavs/00000309-00000300.wav",
}
response = requests.post(url+"/v1/generate", json={**messages, **decode_args})
print("Status Code:", response.status_code)

In [ ]:
# 整合
decode_args = {
    "task_type": "speech",
    "response_mode": "path",
    "max_decode_steps": 200,
    "output_wav_path": "output/service/client.wav",
}
instruction = {
    #"BGM": {"Genre": "当代古典音乐.", "Mood": "温暖 / 友善.", "Instrument": "电吉他", "Theme": "节日.", "SNR": 10.0, "ENV": None},
    "IP": "灵小甄",
    #"风格": "这是一种ASMR耳语，属于一种旨在引发特殊感官体验的创意风格。这个女性使用轻柔的普通话进行耳语(在一侧耳边)，声音气音成分重。音量极低，紧贴麦克风，语速极慢，旨在制造触发听者颅内快感的声学刺激。",
    "语速": "慢速",
    "基频": "中",
    "音量": "中",
    "情感": "高兴",
    "方言": "粤语",

}
messages = {
    "prompt": "Please generate speech based on the following description.\n",
    "text": "我们的愿景是构建未来服务业的数字化基础设施，为世界带来更多微小而美好的改变。",
    "instruction": instruction
}
response = requests.post(url+"/v1/generate", json={**messages, **decode_args})
print("Status Code:", response.status_code)

### Stream策略测试
先做**非流式**（`response_mode=path`，返回JSON）验证；再做**流式下载**（`response_mode=stream` + `requests.stream=True`）验证。

In [ ]:
# 1) 非流式测试（返回JSON + 落盘路径）
import time

payload_non_stream = {
    "task_type": "speech",
    "response_mode": "path",
    "output_wav_path": "output/service/non_stream_test.wav",
    "prompt": "Please generate speech based on the following description.\n",
    "text": "这是非流式测试，用于验证后端返回JSON和音频落盘路径。",
    "max_decode_steps": 120,
}

start = time.time()
resp = requests.post(f"{url}/v1/generate", json=payload_non_stream, timeout=600)
elapsed = time.time() - start

print("Status:", resp.status_code)
print("Elapsed(s):", round(elapsed, 2))
print("Content-Type:", resp.headers.get("content-type"))
if resp.ok:
    print("Body:", resp.json())
else:
    print(resp.text)

In [ ]:
# 2) 流式测试（真实流式：后端逐步生成PCM分片）+ 卡顿判断
import os
import time
import wave

payload_stream = {
    "prompt": "Please generate speech based on the following description.\n",
    "text": "这是流式下载测试，用于验证后端是否支持逐步生成并返回音频流。",
    "max_decode_steps": 120,
}

stream_out = "output/service/stream_test.wav"
os.makedirs(os.path.dirname(stream_out), exist_ok=True)

start = time.time()
first_chunk_time = None
size = 0
chunk_count = 0
pcm_bytes = bytearray()

# 卡顿判断：若相邻两次chunk到达间隔 > 上一个chunk可播放时长，则记为卡顿
stutter_count = 0
max_overrun = 0.0
prev_arrival = None
prev_chunk_duration = None

with requests.post(f"{url}/v1/stream", json=payload_stream, stream=True, timeout=600) as r:
    print("Status:", r.status_code)
    print("Content-Type:", r.headers.get("content-type"))
    sample_rate = int(r.headers.get("X-Audio-Sample-Rate", "24000"))
    codec = r.headers.get("X-Audio-Codec", "")
    print("Codec:", codec)
    print("Sample Rate:", sample_rate)

    if r.status_code != 200:
        print(r.text)
    else:
        for chunk in r.iter_content(chunk_size=8192):
            if not chunk:
                continue

            now = time.time()
            if first_chunk_time is None:
                first_chunk_time = now - start

            if prev_arrival is not None and prev_chunk_duration is not None:
                gap = now - prev_arrival
                overrun = gap - prev_chunk_duration
                if overrun > 0:
                    stutter_count += 1
                    if overrun > max_overrun:
                        max_overrun = overrun

            chunk_duration = len(chunk) / (2.0 * sample_rate)  # mono, int16
            prev_arrival = now
            prev_chunk_duration = chunk_duration

            pcm_bytes.extend(chunk)
            size += len(chunk)
            chunk_count += 1

elapsed = time.time() - start

# 将流式PCM封装为可播放WAV
with wave.open(stream_out, "wb") as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)
    wf.setframerate(sample_rate)
    wf.writeframes(bytes(pcm_bytes))

print("Saved:", stream_out)
print("Total Bytes:", size)
print("Chunk Count:", chunk_count)
print("First Chunk Delay(s):", None if first_chunk_time is None else round(first_chunk_time, 3))
print("Total Elapsed(s):", round(elapsed, 3))
print("Stutter Count:", stutter_count)
print("Max Overrun(s):", round(max_overrun, 4))

if first_chunk_time is None:
    print("结论：没有收到分块数据，流式传输不可用。")
elif first_chunk_time < elapsed * 0.6:
    print("结论：已实现真实流式（首包明显早于整体完成）。")
else:
    print("结论：分块已生效，但首包仍偏晚，可能受模型首轮计算耗时影响。")

if stutter_count > 0:
    print("卡顿提示：检测到chunk供给中断（某个chunk播放完毕后，下一个chunk尚未到达）。")
else:
    print("卡顿提示：未检测到明显chunk级卡顿。")

### RTF测试
先执行预热，再执行RTF计算。RTF定义：`RTF = 生成总耗时 / 生成音频时长`。RTF越小越快（通常 `< 1` 表示快于实时）。

In [3]:
# 3) 预热单元格（建议每次重启服务后先运行）
import time

warmup_payload = {
    "prompt": "Please generate speech based on the following description.\n",
    "text": "warmup",
    "max_decode_steps": 16,
}

warmup_start = time.time()
warmup_bytes = 0
with requests.post(f"{url}/v1/stream", json=warmup_payload, stream=True, timeout=600) as r:
    print("Warmup Status:", r.status_code)
    if r.status_code == 200:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                warmup_bytes += len(chunk)
    else:
        print(r.text)

print("Warmup Bytes:", warmup_bytes)
print("Warmup Elapsed(s):", round(time.time() - warmup_start, 2))

Warmup Status: 200
Warmup Bytes: 420714
Warmup Elapsed(s): 11.83


In [7]:
# 4) RTF计算（基于流式接口）+ 卡顿判断
import os
import time
import wave

rtf_payload = {
    "prompt": "Please generate speech based on the following description.\n",
    "text": "这是RTF测试文本，用于评估流式接口的实时率表现。",
    "max_decode_steps": 120,
}

rtf_out = "output/service/rtf_stream_test.wav"
os.makedirs(os.path.dirname(rtf_out), exist_ok=True)

start = time.time()
first_chunk_time = None
pcm_bytes = bytearray()

stutter_count = 0
max_overrun = 0.0
prev_arrival = None
prev_chunk_duration = None

with requests.post(f"{url}/v1/stream", json=rtf_payload, stream=True, timeout=600) as r:
    print("RTF Test Status:", r.status_code)
    sample_rate = int(r.headers.get("X-Audio-Sample-Rate", "24000"))
    print("Sample Rate:", sample_rate)
    if r.status_code != 200:
        print(r.text)
    else:
        for chunk in r.iter_content(chunk_size=8192):
            if not chunk:
                continue

            now = time.time()
            if first_chunk_time is None:
                first_chunk_time = now - start

            if prev_arrival is not None and prev_chunk_duration is not None:
                gap = now - prev_arrival
                overrun = gap - prev_chunk_duration
                if overrun > 0:
                    stutter_count += 1
                    if overrun > max_overrun:
                        max_overrun = overrun

            chunk_duration = len(chunk) / (2.0 * sample_rate)
            prev_arrival = now
            prev_chunk_duration = chunk_duration

            pcm_bytes.extend(chunk)

gen_elapsed = time.time() - start

with wave.open(rtf_out, "wb") as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)
    wf.setframerate(sample_rate)
    wf.writeframes(bytes(pcm_bytes))

with wave.open(rtf_out, "rb") as rf:
    nframes = rf.getnframes()
    framerate = rf.getframerate()
    audio_duration = nframes / float(framerate) if framerate else 0.0

rtf = (gen_elapsed / audio_duration) if audio_duration > 0 else float("inf")

print("Saved:", rtf_out)
print("First Chunk Delay(s):", None if first_chunk_time is None else round(first_chunk_time, 3))
print("Generation Elapsed(s):", round(gen_elapsed, 3))
print("Audio Duration(s):", round(audio_duration, 3))
print("RTF:", round(rtf, 3))
print("Stutter Count:", stutter_count)
print("Max Overrun(s):", round(max_overrun, 4))

if rtf < 1:
    print("结论：RTF < 1，生成速度快于实时。")
else:
    print("结论：RTF >= 1，生成速度慢于或接近实时。")

if stutter_count > 0:
    print("卡顿提示：检测到chunk供给中断（某个chunk播放完毕后，下一个chunk尚未到达）。")
else:
    print("卡顿提示：未检测到明显chunk级卡顿。")

RTF Test Status: 200
Sample Rate: 44100
Saved: output/service/rtf_stream_test.wav
First Chunk Delay(s): 0.502
Generation Elapsed(s): 4.053
Audio Duration(s): 4.8
RTF: 0.844
Stutter Count: 13
Max Overrun(s): 0.2648
结论：RTF < 1，生成速度快于实时。
卡顿提示：检测到chunk供给中断（某个chunk播放完毕后，下一个chunk尚未到达）。


In [8]:
# 5) RTF计算（缓存池模式：提高首包延迟换取流畅度）
import os
import time
import wave

buffer_payload = {
    "prompt": "Please generate speech based on the following description.\n",
    "text": "这是缓存池RTF测试文本",
    "max_decode_steps": 120,
}

buffer_out = "output/service/rtf_stream_buffered_test.wav"
os.makedirs(os.path.dirname(buffer_out), exist_ok=True)

startup_buffer_sec = 0.52  # 缓存池启动阈值（秒），可调大以换取更平滑

start = time.time()
first_chunk_time = None
playback_started = False
buffer_sec = 0.0
last_event_time = None
stutter_count = 0
max_buffer_deficit = 0.0
pcm_bytes = bytearray()

with requests.post(f"{url}/v1/stream", json=buffer_payload, stream=True, timeout=600) as r:
    print("Buffered RTF Status:", r.status_code)
    sample_rate = int(r.headers.get("X-Audio-Sample-Rate", "24000"))
    print("Sample Rate:", sample_rate)
    print("Startup Buffer(s):", startup_buffer_sec)

    if r.status_code != 200:
        print(r.text)
    else:
        for chunk in r.iter_content(chunk_size=8192):
            if not chunk:
                continue

            now = time.time()
            if first_chunk_time is None:
                first_chunk_time = now - start

            # 到达新chunk前，按时间流逝消耗缓存池
            if playback_started and last_event_time is not None:
                consumed = now - last_event_time
                buffer_sec -= consumed
                if buffer_sec < 0:
                    stutter_count += 1
                    deficit = -buffer_sec
                    if deficit > max_buffer_deficit:
                        max_buffer_deficit = deficit
                    buffer_sec = 0.0

            chunk_duration = len(chunk) / (2.0 * sample_rate)
            buffer_sec += chunk_duration

            # 缓存足够后再开始“播放”
            if (not playback_started) and (buffer_sec >= startup_buffer_sec):
                playback_started = True

            last_event_time = now
            pcm_bytes.extend(chunk)

gen_elapsed = time.time() - start

with wave.open(buffer_out, "wb") as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)
    wf.setframerate(sample_rate)
    wf.writeframes(bytes(pcm_bytes))

with wave.open(buffer_out, "rb") as rf:
    nframes = rf.getnframes()
    framerate = rf.getframerate()
    audio_duration = nframes / float(framerate) if framerate else 0.0

rtf = (gen_elapsed / audio_duration) if audio_duration > 0 else float("inf")

print("Saved:", buffer_out)
print("First Chunk Delay(s):", None if first_chunk_time is None else round(first_chunk_time, 3))
print("Generation Elapsed(s):", round(gen_elapsed, 3))
print("Audio Duration(s):", round(audio_duration, 3))
print("RTF:", round(rtf, 3))
print("Playback Started:", playback_started)
print("Stutter Count:", stutter_count)
print("Max Buffer Deficit(s):", round(max_buffer_deficit, 4))

if rtf < 1:
    print("结论：RTF < 1，生成速度快于实时。")
else:
    print("结论：RTF >= 1，生成速度慢于或接近实时。")

if stutter_count > 0:
    print("卡顿提示：检测到chunk供给中断（某个chunk播放完毕后，下一个chunk尚未到达）。")
else:
    print("卡顿提示：未检测到明显chunk级卡顿。")

Buffered RTF Status: 200
Sample Rate: 44100
Startup Buffer(s): 0.52
Saved: output/service/rtf_stream_buffered_test.wav
First Chunk Delay(s): 0.481
Generation Elapsed(s): 2.624
Audio Duration(s): 3.2
RTF: 0.82
Playback Started: True
Stutter Count: 0
Max Buffer Deficit(s): 0.0
结论：RTF < 1，生成速度快于实时。
卡顿提示：未检测到明显chunk级卡顿。
